# Yasmin tutorial

This notebook introduces the basic Yasmin workflow: define fields, write a stencil, build an operator, execute it, and inspect the generated intermediate representations.

From the repository root, install Yasmin first with `python -m pip install -e .`.


In [ ]:
import numpy as np

import yasmin as yasi

## 1. Describe the grid and data

`Dimension`, `Field`, and `Scalar` objects describe the symbolic inputs to a computation. They do not allocate arrays themselves.


In [ ]:
x, y = yasi.Dimension("x", "y")

u = yasi.Field("u", dims=(x, y), dtype=yasi.float64)
out = yasi.Field("out", dims=(x, y), dtype=yasi.float64)
alpha = yasi.Scalar("alpha", dtype=yasi.float64)

## 2. Define a stencil and operator

Field indices are offsets from the current grid point. For example, `f[-1, 0]` reads the neighboring point one position away along the first dimension.


In [ ]:
@yasi.stencil
def laplace(f: yasi.Field) -> yasi.SymbolicExpr:
    return f[-1, 0] + f[1, 0] + f[0, -1] + f[0, 1] - 4.0 * f[0, 0]


@yasi.operator
def diffuse(out: yasi.Field, u: yasi.Field, alpha: yasi.Scalar) -> None:
    out[0, 0] = u[0, 0] + alpha * laplace(u)


op = diffuse(out, u, alpha)

## 3. Execute with NumPy

Runtime arrays are bound to symbolic fields through the `fields` mapping. Yasmin infers the one-cell halo required by the Laplacian, so the outer boundary remains untouched.


In [ ]:
u_data = np.zeros((8, 8), dtype=np.float64)
u_data[3:5, 3:5] = 1.0
out_data = np.zeros_like(u_data)

yasi.execute(
    op,
    backend="numpy",
    fields={u: u_data, out: out_data},
    scalars={alpha: 0.1},
)

out_data

## 4. Inspect the compiler IR

The Stencil IR represents the symbolic computation. Native backends lower it further to explicit loop nests.


In [ ]:
yasi.print_stencil_ir(op)
yasi.print_loop_ir(op)

## 5. Native backends

When a compatible compiler is installed, the same operator can run with the C++ or OpenMP backend. A `CompileConfig` lets you select backend-specific options, and the returned kernel can be reused.


In [ ]:
openmp_config = yasi.CompileConfig(
    backend="openmp",
    options=yasi.OpenMPOptions(
        num_threads=4,
        schedule="static",
    ),
)

# Example for a machine with a compatible OpenMP compiler:
# kernel = yasi.compile(op, config=openmp_config)
# kernel(fields={u: u_data, out: out_data}, scalars={alpha: 0.1})
# print(kernel.source)

That is the basic Yasmin workflow. The implementation details live in `src/yasmin`, and `docs/design.md` describes the compiler architecture in more detail.
